In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!nvidia-smi

In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets transformers deepspeed
!pip install -q trl

print("✅ Dependencies installed")

In [ ]:
# Cell 2: Create sample preference pairs
import json
import os

os.makedirs("data", exist_ok=True)

sample_pairs = [
    {
        "prompt": "Movie: The Conjuring\nOverview: Paranormal investigators help a family terrorized by a dark presence.\nGenres: Horror, Mystery, Thriller\nRating: 7.5\n\nIs this movie appropriate for children aged 5-10?",
        "chosen": "Classification: Not safe for children\nExplanation: This movie is in the Horror genre and contains frightening supernatural imagery that is not appropriate for children.",
        "rejected": "Classification: Safe for children\nExplanation: The movie has a high rating."
    },
    {
        "prompt": "Movie: Finding Nemo\nOverview: A clownfish sets out on a journey to find his son.\nGenres: Animation, Adventure, Comedy\nRating: 8.2\n\nIs this movie appropriate for children aged 5-10?",
        "chosen": "Classification: Safe for children\nExplanation: This is an animated, family-friendly film with positive messages about family and friendship.",
        "rejected": "Classification: Not safe for children\nExplanation: The movie has some intense scenes."
    },
    {
        "prompt": "Movie: Jurassic Park\nOverview: A paleontologist must protect kids after the park's cloned dinosaurs run loose.\nGenres: Action, Adventure, Sci-Fi\nRating: 8.2\n\nIs this movie appropriate for children aged 5-10?",
        "chosen": "Classification: Not safe for children\nExplanation: Jurassic Park contains intense dinosaur attacks and frightening imagery that may scare young children.",
        "rejected": "Classification: Safe for children\nExplanation: It's a popular adventure movie."
    },
    {
        "prompt": "Movie: Toy Story\nOverview: A cowboy doll is threatened when a new spaceman figure supplants him as top toy.\nGenres: Animation, Adventure, Comedy\nRating: 8.3\n\nIs this movie appropriate for children aged 5-10?",
        "chosen": "Classification: Safe for children\nExplanation: This is an animated, family-friendly film with positive messages about friendship.",
        "rejected": "Classification: Not safe for children\nExplanation: There are some tense moments."
    },
    {
        "prompt": "Movie: The Dark Knight\nOverview: When the Joker wreaks havoc on Gotham, Batman must accept one of the greatest tests of his ability.\nGenres: Action, Crime, Drama\nRating: 9.0\n\nIs this movie appropriate for children aged 5-10?",
        "chosen": "Classification: Not safe for children\nExplanation: The Dark Knight contains intense violence, disturbing imagery, and adult themes not suitable for young children.",
        "rejected": "Classification: Safe for children\nExplanation: It's a superhero movie."
    },
    {
        "prompt": "Movie: Moana\nOverview: A young woman uses her navigational talents to set sail for a fabled island.\nGenres: Animation, Adventure, Comedy\nRating: 7.6\n\nIs this movie appropriate for children aged 5-10?",
        "chosen": "Classification: Safe for children\nExplanation: Moana is an animated, family-friendly film with positive messages about courage and self-discovery.",
        "rejected": "Classification: Not safe for children\nExplanation: There is some action."
    },
]

with open("data/preference_pairs.jsonl", "w") as f:
    for pair in sample_pairs:
        f.write(json.dumps(pair) + "\n")

print(f"✅ Created {len(sample_pairs)} preference pairs")

In [ ]:
# Cell 3: Verify GPU setup
import torch

print(f"Number of GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

print(f"\n✅ Total VRAM: {sum(torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count())) / 1e9:.1f} GB")

In [ ]:
# Cell 5: Load model and dataset
from unsloth import FastLanguageModel, PatchDPOTrainer
from datasets import load_dataset
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

PatchDPOTrainer()

# Load dataset
dataset = load_dataset(
    "json",
    data_files="data/preference_pairs.jsonl",
    split="train"
)

print(f"✅ Loaded {len(dataset)} preference pairs")

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/zephyr-sft-bnb-4bit",
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)

print("✅ Model loaded")

In [ ]:
# Cell 6: Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=1024,
)

print("✅ LoRA adapters added")

In [ ]:
# Cell 7: Configure DPO trainer WITH DeepSpeed
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=DPOConfig(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        warmup_ratio=0.1,
        num_train_epochs=2,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        seed=42,
        output_dir="outputs_dpo_multi_gpu",
        report_to="none",
        # ✅ DEEPSPEED CONFIGURATION
        deepspeed="ds_config.json",
    ),
    beta=0.1,
    train_dataset=dataset,
    tokenizer=tokenizer,
    max_length=1024,
    max_prompt_length=512,
)

print("✅ DPO Trainer configured with DeepSpeed")

In [ ]:
# Verificar se o arquivo existe
import os
print("Current directory:", os.getcwd())
print("ds_config.json exists:", os.path.exists("ds_config.json"))
print("Files in current dir:", os.listdir(".")[:10])

In [ ]:
# Recriar o arquivo com caminho absoluto
import json
import os

# Use absolute path
CONFIG_PATH = "/kaggle/working/ds_config.json"

deepspeed_config = {
    "train_batch_size": "auto",
    "train_micro_batch_size_per_gpu": "auto",
    "gradient_accumulation_steps": "auto",
    "gradient_clipping": "auto",
    "zero_allow_untested_optimizer": True,
    "bf16": {"enabled": "auto"},
    "optimizer": {
        "type": "AdamW",
        "params": {
            "lr": "auto",
            "betas": "auto",
            "eps": "auto",
            "weight_decay": "auto"
        }
    },
    "scheduler": {
        "type": "WarmupLR",
        "params": {
            "warmup_min_lr": "auto",
            "warmup_max_lr": "auto",
            "warmup_num_steps": "auto"
        }
    },
    "zero_optimization": {
        "stage": 2,
        "offload_optimizer": {"device": "none"},
        "allgather_partitions": True,
        "allgather_bucket_size": 2e8,
        "overlap_comm": True,
        "reduce_scatter": True,
        "reduce_bucket_size": 2e8,
        "contiguous_gradients": True
    }
}

with open(CONFIG_PATH, "w") as f:
    json.dump(deepspeed_config, f, indent=2)

print(f"✅ DeepSpeed config saved to: {CONFIG_PATH}")
print(f"File exists: {os.path.exists(CONFIG_PATH)}")

In [ ]:
# Agora use o caminho absoluto
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=DPOConfig(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        warmup_ratio=0.1,
        num_train_epochs=2,
        fp16=False,
        bf16=True,
        logging_steps=1,
        optim="adamw_8bit",
        seed=42,
        output_dir="outputs_dpo_multi_gpu",
        report_to="none",
        # ✅ USE ABSOLUTE PATH
        deepspeed="/kaggle/working/ds_config.json",
    ),
    beta=0.1,
    train_dataset=dataset,
    tokenizer=tokenizer,
    max_length=1024,
    max_prompt_length=512,
)

print("✅ DPO Trainer configured with DeepSpeed")

In [ ]:
# Substitua esta célula:
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=DPOConfig(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        warmup_ratio=0.1,
        num_train_epochs=2,
        # ✅ FORCE FP16 (T4 doesn't support bf16)
        fp16=True,
        bf16=False,
        logging_steps=1,
        optim="adamw_8bit",
        seed=42,
        output_dir="outputs_dpo_multi_gpu",
        report_to="none",
        deepspeed="/kaggle/working/ds_config.json",
    ),
    beta=0.1,
    train_dataset=dataset,
    tokenizer=tokenizer,
    max_length=1024,
    max_prompt_length=512,
)

print("✅ DPO Trainer configured with DeepSpeed + FP16")

In [ ]:
dpo_trainer.train()

In [ ]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes deepspeed

In [1]:
import torch
print(f"GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

GPUs: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [2]:
import json
import os

os.makedirs("data", exist_ok=True)

# Same sample data as before
sample_pairs = [
    {
        "prompt": "Movie: The Conjuring\nOverview: Paranormal investigators help a family terrorized by a dark presence.\nGenres: Horror, Mystery, Thriller\nRating: 7.5\n\nIs this movie appropriate for children aged 5-10?",
        "chosen": "Classification: Not safe for children\nExplanation: This movie is in the Horror genre and contains frightening supernatural imagery that is not appropriate for children.",
        "rejected": "Classification: Safe for children\nExplanation: The movie has a high rating."
    },
    {
        "prompt": "Movie: Finding Nemo\nOverview: A clownfish sets out on a journey to find his son.\nGenres: Animation, Adventure, Comedy\nRating: 8.2\n\nIs this movie appropriate for children aged 5-10?",
        "chosen": "Classification: Safe for children\nExplanation: This is an animated, family-friendly film with positive messages about family and friendship.",
        "rejected": "Classification: Not safe for children\nExplanation: The movie has some intense scenes."
    },
    {
        "prompt": "Movie: Jurassic Park\nOverview: A paleontologist must protect kids after the park's cloned dinosaurs run loose.\nGenres: Action, Adventure, Sci-Fi\nRating: 8.2\n\nIs this movie appropriate for children aged 5-10?",
        "chosen": "Classification: Not safe for children\nExplanation: Jurassic Park contains intense dinosaur attacks and frightening imagery that may scare young children.",
        "rejected": "Classification: Safe for children\nExplanation: It's a popular adventure movie."
    },
]

with open("data/preference_pairs.jsonl", "w") as f:
    for pair in sample_pairs:
        f.write(json.dumps(pair) + "\n")

print(f"✅ Created {len(sample_pairs)} preference pairs")

✅ Created 3 preference pairs


In [3]:
import json

config_path = "/kaggle/working/ds_config.json"

deepspeed_config = {
    "train_batch_size": "auto",
    "train_micro_batch_size_per_gpu": "auto",
    "gradient_accumulation_steps": "auto",
    "gradient_clipping": "auto",
    "zero_allow_untested_optimizer": True,
    "fp16": {
        "enabled": "auto",
        "loss_scale": 0,
        "loss_scale_window": 1000,
        "initial_scale_power": 16,
        "hysteresis": 2,
        "min_loss_scale": 1
    },
    "bf16": {"enabled": False},
    "optimizer": {
        "type": "AdamW",
        "params": {
            "lr": "auto",
            "betas": "auto",
            "eps": "auto",
            "weight_decay": "auto"
        }
    },
    "scheduler": {
        "type": "WarmupLR",
        "params": {
            "warmup_min_lr": "auto",
            "warmup_max_lr": "auto",
            "warmup_num_steps": "auto"
        }
    },
    "zero_optimization": {
        "stage": 2,
        "offload_optimizer": {"device": "none"},
        "allgather_partitions": True,
        "allgather_bucket_size": 2e8,
        "overlap_comm": True,
        "reduce_scatter": True,
        "reduce_bucket_size": 2e8,
        "contiguous_gradients": True
    }
}

with open(config_path, "w") as f:
    json.dump(deepspeed_config, f, indent=2)

print(f"✅ Config saved: {config_path}")

✅ Config saved: /kaggle/working/ds_config.json


In [4]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="data/preference_pairs.jsonl",
    split="train"
)

print(f"✅ Loaded {len(dataset)} pairs")
print(f"Columns: {dataset.column_names}")

Generating train split: 0 examples [00:00, ? examples/s]

✅ Loaded 3 pairs
Columns: ['prompt', 'chosen', 'rejected']


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

# ✅ NO device_map — DeepSpeed will handle distribution
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small for testing

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    # ❌ NO device_map — let DeepSpeed handle it
)

# Add LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("✅ Model loaded with LoRA (no device_map)")

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079
✅ Model loaded with LoRA (no device_map)


In [6]:
def format_prompt(example):
    """Format prompt for DPO training."""
    return {
        "prompt": example["prompt"],
        "chosen": example["chosen"],
        "rejected": example["rejected"],
    }

dataset = dataset.map(format_prompt)
print("✅ Dataset formatted")

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

✅ Dataset formatted


In [7]:
from trl import DPOTrainer, DPOConfig

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=DPOConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.1,
        num_train_epochs=1,
        fp16=True,
        bf16=False,
        logging_steps=1,
        optim="adamw_torch",
        seed=42,
        output_dir="outputs_dpo_multi_gpu",
        report_to="none",
        deepspeed="/kaggle/working/ds_config.json",
        remove_unused_columns=False,
    ),
    beta=0.1,
    train_dataset=dataset,
    tokenizer=tokenizer,
    max_length=512,
    max_prompt_length=256,
)

print("✅ DPO Trainer configured")

TypeError: DPOConfig.__init__() got an unexpected keyword argument 'warmup_ratio'

In [12]:
from trl import DPOTrainer, DPOConfig

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=DPOConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        # ❌ REMOVED: warmup_ratio=0.1,
        warmup_steps=10,  # ✅ Use steps instead of ratio
        num_train_epochs=1,
        fp16=True,
        bf16=False,
        logging_steps=1,
        optim="adamw_torch",
        seed=42,
        output_dir="outputs_dpo_multi_gpu",
        report_to="none",
        deepspeed="/kaggle/working/ds_config.json",
        remove_unused_columns=False,
    ),
    train_dataset=dataset,
    # tokenizer=tokenizer,
    # max_length=512,
    # max_prompt_length=256,
)

print("✅ DPO Trainer configured")

/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


Adding EOS to train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

✅ DPO Trainer configured


In [13]:
dpo_trainer.train()

[RANK 0] Gradient accumulation steps mismatch: GradientAccumulationPlugin has 1, DeepSpeed config has 4. Using DeepSpeed's value.


RuntimeError: module must have its parameters and buffers on device cuda:0 (device_ids[0]) but found one of them on device: cpu

In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# ✅ Load model and move to GPU 0
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map=None,  # ← NO auto device_map
)

# ✅ Move to cuda:0 explicitly
model = model.to("cuda:0")

# Add LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("✅ Model loaded and moved to GPU 0")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079
✅ Model loaded and moved to GPU 0


In [17]:
import os

# ✅ Force single-process mode (no distributed training)
os.environ["ACCELERATE_USE_DEEPSPEED"] = "false"
os.environ["ACCELERATE_USE_FSDP"] = "false"
os.environ["RANK"] = "0"
os.environ["LOCAL_RANK"] = "0"
os.environ["WORLD_SIZE"] = "1"
os.environ["MASTER_ADDR"] = "127.0.0.1"
os.environ["MASTER_PORT"] = "29500"

print("✅ Environment set for single-process mode")

✅ Environment set for single-process mode


In [18]:
from trl import DPOTrainer, DPOConfig

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=DPOConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=1,
        fp16=True,
        bf16=False,
        logging_steps=1,
        optim="adamw_torch",
        seed=42,
        output_dir="outputs_dpo_multi_gpu",
        report_to="none",
        # ❌ NO deepspeed config
    ),
    # beta=0.1,
    train_dataset=dataset,
    # tokenizer=tokenizer,
    # max_length=512,
    # max_prompt_length=256,
)

print("✅ DPO Trainer configured (DDP mode)")

Dropping fully truncated examples from train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

✅ DPO Trainer configured (DDP mode)


In [19]:
dpo_trainer.train()

Step,Training Loss
1,0.693147


TrainOutput(global_step=1, training_loss=0.6931471824645996, metrics={'train_runtime': 2.4735, 'train_samples_per_second': 1.213, 'train_steps_per_second': 0.404, 'total_flos': 3731640287232.0, 'train_loss': 0.6931471824645996, 'epoch': 1.0})